In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import joblib

df = pd.read_csv("../data/cleaned.csv")

# Normalize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Dynamically find columns
streams_col = next((c for c in df.columns if "stream" in c), None)
dance_col = next((c for c in df.columns if "dance" in c), None)
energy_col = next((c for c in df.columns if "energy" in c), None)

# Ensure numeric
for col in [streams_col, dance_col, energy_col]:
    if col:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=[c for c in [streams_col, dance_col, energy_col] if c])

# Target
df['hit'] = (df[streams_col] > df[streams_col].median()).astype(int)

# Feature Engineering (safe)
if energy_col and dance_col:
    df['energy_dance'] = df[energy_col] * df[dance_col]

# Select features
X = df.select_dtypes(include=['float64', 'int64']).drop(['hit'], axis=1)
y = df['hit']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Models
lr = LogisticRegression(max_iter=1000)
dt = DecisionTreeClassifier()

lr.fit(X_train, y_train)
dt.fit(X_train, y_train)

# Evaluation
lr_pred = lr.predict(X_test)
dt_pred = dt.predict(X_test)

print("Logistic Regression:", accuracy_score(y_test, lr_pred))
print("Decision Tree:", accuracy_score(y_test, dt_pred))

# Save best model
best_model = dt
joblib.dump(best_model, "../models/supervised_best.pkl")

Logistic Regression: 0.9939024390243902
Decision Tree: 0.9939024390243902


['../models/supervised_best.pkl']